# Warm start share of the slowest calls — HUA (Huawei 2025 public traces)

Port of `notebooks/02_warm_share_of_slowest.ipynb` to the Huawei trace.

- Latency: `(totalCost_frontend - runtimeCost) * 1000` — platform delay in ms
- Warm: every request but the first per `podID`
- One pooled series, 944,522,485 rows in `requests_latency`

Needs a strong machine; the table is 62 GiB and the queries are memory-bound. Writes the curve and the table to
`../../data/tertiary_aggregations/`. The figures are drawn by
`tertiary-repos/warm_share_of_slowest.ipynb`, which reads those CSVs.

In [ ]:
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

DB_PATH = "huawei-data/2025-public-requests/2025-public-requests.duckdb"
TABLE = "requests_latency"

# One region for a quick run (region-4 is 66.8M rows), None for the whole table.
REGION_FILTER = None

AGG_DIR = Path("../../data/tertiary_aggregations")
AGG_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(DB_PATH, read_only=True)
con.execute("PRAGMA threads=30")
con.execute("PRAGMA memory_limit='250GB'")
con.execute("PRAGMA temp_directory='./duckdb_temp'")

print(f"duckdb {duckdb.__version__}  numpy {np.__version__}  pandas {pd.__version__}")
print(f"DB: {Path(DB_PATH).resolve()}")
print(f"REGION_FILTER = {REGION_FILTER!r}")

In [ ]:
# Same grids and truncation floor as the source notebook.
PERCENTILES = np.linspace(0.80, 1.00, 10000)
TABLE_PERCENTILES = np.concatenate([
    np.array([0.80, 0.85, 0.90, 0.95]),
    np.linspace(0.97, 1.00, 61),
])
MIN_N_FOR_CURVE = 30


def warm_share_curve(latency, is_warm, percentiles):
    """% of calls at or above each percentile threshold that are warm, and how many that is.

    Sorts once and reads suffix sums off the sorted order instead of rescanning the array per
    percentile, which at this row count would not finish. Exact, not an approximation.
    """
    latency = np.asarray(latency, dtype=float)
    is_warm = np.asarray(is_warm, dtype=bool)
    percentiles = np.asarray(percentiles, dtype=float)
    n_total = latency.size
    if n_total == 0:
        nan = np.full(percentiles.size, np.nan)
        return nan, np.zeros(percentiles.size, dtype=int), nan

    order = np.argsort(latency, kind="stable")
    lat_sorted = latency[order]
    warm_sorted = is_warm[order]
    del order

    suffix_warm = np.empty(n_total + 1, dtype=np.int64)
    suffix_warm[:n_total] = np.cumsum(warm_sorted[::-1])[::-1]
    suffix_warm[n_total] = 0
    del warm_sorted

    # 'linear' matches pandas' Series.quantile default.
    thresholds = np.quantile(lat_sorted, percentiles, method="linear")
    idx = np.searchsorted(lat_sorted, thresholds, side="left")
    n = (n_total - idx).astype(np.int64)
    share = np.where(n > 0, suffix_warm[idx] / np.maximum(n, 1) * 100.0, np.nan)
    return share, n, thresholds


print("metric helpers defined")

In [ ]:
# (latency, is_cold_flag) for every call. No cold filter -- traffic as observed.
where_region = f"      AND region = '{REGION_FILTER}'\n" if REGION_FILTER else ""

query = f"""
WITH parsed AS (
    SELECT
        (totalCost_frontend - runtimeCost) * 1000.0 AS lat_ms,
        ROW_NUMBER() OVER (PARTITION BY podID ORDER BY time_frontend ASC) AS rn
    FROM {TABLE}
    WHERE totalCost_frontend IS NOT NULL
      AND runtimeCost IS NOT NULL
      AND time_frontend IS NOT NULL
      AND podID IS NOT NULL
{where_region})
SELECT lat_ms, (rn > 1) AS is_warm
FROM parsed
WHERE lat_ms >= 0   -- negative values are clock/logging drift
"""

print(query)

t0 = time.time()
# fetch_arrow_table, not .arrow(), which returns a RecordBatchReader on duckdb >= 1.5
tbl = con.execute(query).fetch_arrow_table()
print(f"query + fetch: {time.time() - t0:,.1f}s, {tbl.num_rows:,} rows")

t0 = time.time()
latency = tbl.column("lat_ms").to_numpy(zero_copy_only=False).astype(np.float64, copy=False)
is_warm = tbl.column("is_warm").to_numpy(zero_copy_only=False).astype(bool, copy=False)
del tbl
print(f"to numpy: {time.time() - t0:,.1f}s, "
      f"{latency.nbytes / 1e9:,.2f} GB + {is_warm.nbytes / 1e9:,.2f} GB")

In [ ]:
GROUP = "huawei"
LABEL = "Huawei25"

baseline_cold_pct = float((~is_warm).mean() * 100.0)
n_total = int(latency.size)
n_cold = int((~is_warm).sum())

print(f"n_total            = {n_total:,}")
print(f"cold (first/pod)   = {n_cold:,}")
print(f"baseline_cold_pct  = {baseline_cold_pct:.6f}%")
print(f"median lat_ms      = {np.median(latency):,.4f}")

t0 = time.time()
share_c, n_c, thr_c = warm_share_curve(latency, is_warm, PERCENTILES)
print(f"curve grid ({PERCENTILES.size:,} points): {time.time() - t0:,.1f}s")

t0 = time.time()
share_t, n_t, thr_t = warm_share_curve(latency, is_warm, TABLE_PERCENTILES)
print(f"table grid ({TABLE_PERCENTILES.size} points): {time.time() - t0:,.1f}s")

df_curve = pd.DataFrame({
    "dataset": "HUA",
    "platform": GROUP,
    "label": LABEL,
    "percentile": PERCENTILES * 100.0,
    "pct_warm": share_c,
    "n": n_c,
    "threshold_ms": thr_c,
    "baseline_cold_pct": baseline_cold_pct,
})
df_curve["pct_cold"] = 100.0 - df_curve["pct_warm"]
df_curve["cold_enrichment"] = df_curve["pct_cold"] / df_curve["baseline_cold_pct"]

df_table = pd.DataFrame({
    "dataset": "HUA",
    "platform": GROUP,
    "label": LABEL,
    "percentile": np.round(TABLE_PERCENTILES * 100.0, 2),
    "top_pct": np.round(100.0 - TABLE_PERCENTILES * 100.0, 2),
    "slowest_n": n_t,
    "pct_warm": share_t,
    "threshold_ms": thr_t,
    "baseline_cold_pct": baseline_cold_pct,
})
df_table["pct_cold"] = 100.0 - df_table["pct_warm"]
df_table["cold_enrichment"] = df_table["pct_cold"] / df_table["baseline_cold_pct"]

below = df_curve.loc[df_curve["n"] < MIN_N_FOR_CURVE, "percentile"]
cut = float(below.min()) if not below.empty else 100.0
print(f"\nMIN_N_FOR_CURVE={MIN_N_FOR_CURVE} truncates the curve at p{cut:.4f}")
print(f"n at p99.99 = {int(np.interp(99.99, df_curve['percentile'], df_curve['n'])):,}")

print("\n" + "=" * 96)
print(f"HUA / {LABEL}: warm-start share among the slowest calls")
print("=" * 96)
print(df_table.drop(columns=["dataset", "platform", "label"]).round(4).to_string(index=False))

In [ ]:
# Per-config breakdown (mc / mem out of podID). Not used by the cross-dataset figure, which
# carries one series per platform.
#
# podID looks like `pool22-300-128-0001573974`; both fields are required to be all-digits.
config_query = f"""
WITH parsed AS (
    SELECT
        (totalCost_frontend - runtimeCost) * 1000.0 AS lat_ms,
        split_part(podID, '-', 2) AS mc,
        split_part(podID, '-', 3) AS mem,
        ROW_NUMBER() OVER (PARTITION BY podID ORDER BY time_frontend ASC) AS rn
    FROM {TABLE}
    WHERE totalCost_frontend IS NOT NULL
      AND runtimeCost IS NOT NULL
      AND time_frontend IS NOT NULL
      AND podID IS NOT NULL
{where_region})
SELECT
    mc || 'mc / ' || mem || 'MB' AS config,
    count(*) AS n,
    sum(CASE WHEN rn > 1 THEN 0 ELSE 1 END) AS n_cold,
    100.0 * sum(CASE WHEN rn > 1 THEN 0 ELSE 1 END) / count(*) AS baseline_cold_pct,
    quantile_cont(lat_ms, 0.50) AS p50_ms,
    quantile_cont(lat_ms, 0.95) AS p95_ms,
    quantile_cont(lat_ms, 0.99) AS p99_ms,
    quantile_cont(lat_ms, 0.999) AS p999_ms,
    quantile_cont(lat_ms, 0.9999) AS p9999_ms
FROM parsed
WHERE lat_ms >= 0
  AND regexp_matches(mc, '^[0-9]+$')
  AND regexp_matches(mem, '^[0-9]+$')
GROUP BY config
HAVING count(*) >= 1000
ORDER BY n DESC
"""

t0 = time.time()
df_config = con.execute(config_query).df()
print(f"per-config aggregate: {time.time() - t0:,.1f}s, {len(df_config)} configs")
print(df_config.round(4).to_string(index=False))

In [ ]:
curve_path = AGG_DIR / "hua_warm_share_of_slowest_curve.csv"
table_path = AGG_DIR / "hua_warm_share_of_slowest.csv"
config_path = AGG_DIR / "hua_warm_share_by_config.csv"

df_curve.to_csv(curve_path, index=False)
df_table.to_csv(table_path, index=False)
df_config.to_csv(config_path, index=False)

for p in (curve_path, table_path, config_path):
    print(f"wrote {p}  ({p.stat().st_size / 1024:,.1f} KiB)")

con.close()